# GA4 交易号覆盖守护审计

## tl;dr

本 Notebook 审计 v2 GA4 日指标与异常检测。14 天中共有 648 个 `purchase` 事件、190 个有效交易号 purchase 事件；11 月 1–10 日的有效交易号覆盖率均为 0%。因此，所有 56 条依赖规范订单键的日期×指标结果均被标记为**不可作业务解释**，且 v2 没有生成任何渠道/设备贡献记录。20 条统计异常中，订单相关异常只能作为数据覆盖诊断。

## Context & Methods

### Key Assumptions

- 输入批次：`9a2d8d7c-0f5b-4c57-a54d-98c59b9c5a27`；日指标运行：`8a9acff4-fa36-4921-b1c0-8d8fe4f2ef9d`；异常运行：`e9975d26-b3c2-4702-b857-622b6c7da4c1`。
- 有效交易号必须非空、非空白且不为 `(not set)`/`unknown`。覆盖率按 purchase 事件计算；规范订单另按有效交易号去重，二者不能混用。
- 守护阈值为 95%。当前日或此前 7 天滚动基线中只要任一日低于阈值，规范订单会话、规范订单、收入与结账至规范订单率只能用于数据质量诊断，不得作为业务异常解释或生成贡献切片。
- MAD 检测仍保留结果以供审计：此前最多 7 天、至少 5 个历史点、稳健 Z 阈值 3.5。

In [1]:
from pathlib import Path
import os

import pandas as pd
import pymysql
from dotenv import load_dotenv

project_root = Path.cwd().resolve()
if not (project_root / '.env').exists():
    project_root = project_root.parent
load_dotenv(project_root / '.env')

BATCH_ID = '9a2d8d7c-0f5b-4c57-a54d-98c59b9c5a27'
DAILY_RUN_ID = '8a9acff4-fa36-4921-b1c0-8d8fe4f2ef9d'
ANOMALY_RUN_ID = 'e9975d26-b3c2-4702-b857-622b6c7da4c1'

connection = pymysql.connect(
    host=os.environ.get('MYSQL_HOST', '127.0.0.1'),
    port=int(os.environ.get('MYSQL_PORT', '3306')),
    user=os.environ.get('MYSQL_USER', 'root'),
    password=os.environ['MYSQL_ROOT_PASSWORD'],
)

## Data

### 1. Load v2 Daily Metrics and Detection Results

In [2]:
daily = pd.read_sql(
    """
    SELECT metric_date, observed_sessions, checkout_start_sessions, purchase_events,
           valid_transaction_id_purchase_events, valid_transaction_id_coverage,
           canonical_orders, canonical_revenue_usd
    FROM insightops_mart.ga4_daily_metric
    WHERE source_batch_id = %s AND definition_version = 'ga4_daily_metrics_v2'
    ORDER BY metric_date
    """,
    connection,
    params=(BATCH_ID,),
)
daily['metric_date'] = pd.to_datetime(daily['metric_date'])
anomalies = pd.read_sql(
    """
    SELECT metric_id, metric_date, metric_value, baseline_points, status,
           is_business_interpretable, interpretation_note
    FROM insightops_ops.metric_anomaly_result
    WHERE run_id = %s
    ORDER BY metric_id, metric_date
    """,
    connection,
    params=(ANOMALY_RUN_ID,),
)
daily[['metric_date', 'purchase_events', 'valid_transaction_id_purchase_events', 'valid_transaction_id_coverage']]

/var/folders/vx/ctbyqz_x2g7169l40g5971w40000gn/T/ipykernel_9004/3527601010.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  daily = pd.read_sql(
/var/folders/vx/ctbyqz_x2g7169l40g5971w40000gn/T/ipykernel_9004/3527601010.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  anomalies = pd.read_sql(


,metric_date,purchase_events,valid_transaction_id_purchase_events,valid_transaction_id_coverage
0,2020-11-01,14,0,0.000000
1,2020-11-02,49,0,0.000000
2,2020-11-03,49,0,0.000000
3,2020-11-04,46,0,0.000000
4,2020-11-05,26,0,0.000000
5,2020-11-06,58,0,0.000000
6,2020-11-07,31,0,0.000000
7,2020-11-08,32,0,0.000000
8,2020-11-09,52,0,0.000000
9,2020-11-10,64,0,0.000000


## Results

### 2. Reconcile Coverage, Guardrail, and Contribution Suppression

In [3]:
canonical_metrics = {
    'ga4_canonical_order_sessions',
    'ga4_canonical_orders',
    'ga4_canonical_revenue_usd',
    'ga4_checkout_to_canonical_order_rate',
}

assert len(daily) == 14
assert int(daily['purchase_events'].sum()) == 648
assert int(daily['valid_transaction_id_purchase_events'].sum()) == 190
assert (daily.loc[daily['metric_date'] <= '2020-11-10', 'valid_transaction_id_coverage'] == 0).all()
assert len(anomalies) == 112

canonical_results = anomalies[anomalies['metric_id'].isin(canonical_metrics)]
coverage_results = anomalies[anomalies['metric_id'].eq('ga4_valid_transaction_id_coverage')]
assert len(canonical_results) == 56
assert not canonical_results['is_business_interpretable'].astype(bool).any()
assert len(coverage_results) == 14
assert not coverage_results['is_business_interpretable'].astype(bool).any()

contribution_count = pd.read_sql(
    """
    SELECT COUNT(*) AS contribution_rows
    FROM insightops_ops.metric_anomaly_contribution AS contribution
    JOIN insightops_ops.metric_anomaly_result AS anomaly
      ON anomaly.anomaly_id = contribution.anomaly_id
    WHERE anomaly.run_id = %s
    """,
    connection,
    params=(ANOMALY_RUN_ID,),
).iloc[0, 0]
assert int(contribution_count) == 0

summary = pd.DataFrame({
    'check': [
        '日指标与 Staging 购买事件总量',
        '日指标与 Staging 有效交易号事件总量',
        '规范订单依赖结果可作业务解释的数量',
        '交易号覆盖守护指标可作业务解释的数量',
        'v2 渠道/设备贡献记录',
    ],
    'observed_value': [648, 190, int(canonical_results['is_business_interpretable'].sum()), int(coverage_results['is_business_interpretable'].sum()), int(contribution_count)],
    'expected_value': [648, 190, 0, 0, 0],
})
summary

/var/folders/vx/ctbyqz_x2g7169l40g5971w40000gn/T/ipykernel_9004/3022850642.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  contribution_count = pd.read_sql(


,check,observed_value,expected_value
0,日指标与 Staging 购买事件总量,648,648
1,日指标与 Staging 有效交易号事件总量,190,190
2,规范订单依赖结果可作业务解释的数量,0,0
3,交易号覆盖守护指标可作业务解释的数量,0,0
4,v2 渠道/设备贡献记录,0,0


## Takeaways

- **高严重度数据质量问题已被系统性隔离。** 11 月 1–10 日 421 个 purchase 事件没有任何有效交易号；若不加守护，订单、收入和转化率会把字段覆盖变化误读为业务增长。
- **v2 保留检测、抑制解释。** 20 条统计异常仍可用于定位覆盖突变；56 条规范订单相关日期×指标结果全部 `is_business_interpretable = 0`，v2 不生成贡献记录。
- **适用边界。** 95% 是当前项目的保守默认阈值。未来接入具有稳定交易号的完整数据后，应重跑同一流程并根据数据合同、迟到数据与业务容忍度复核阈值；短于 5 个历史点的日期依然只标记为 `INSUFFICIENT_HISTORY`。

In [4]:
connection.close()